# Privacy assessment & red-teaming (privacy MCP tools)

Build a small clinical cohort and run the privacy server's detection, assessment, membership-inference and re-identification tools — the same functions the agent calls as `privacy__*` MCP tools.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # repo root (kernel cwd)

from examples.privacy.clinical_cohort import build_cohort
from mcp_servers import privacy_tools as pt

cohort = build_cohort(200, seed=7)
cohort.to_csv('examples/privacy/clinical_cohort.csv', index=False)
print(cohort.shape)

In [ ]:
print(pt.detect_pii_in_text(
    'Contact d.smith@example.org or +1 555-010-1234. Card 4111-1111-1111-1111. '
    'SSN 123-45-6789.'))

In [ ]:
print(pt.assess_dataframe_privacy('examples/privacy/clinical_cohort.csv'))

In [ ]:
print(pt.privacy_redteam_checklist('clinical cohort', has_model=False, public_release=True))
import numpy as np
rng = np.random.default_rng(7)
preds = np.concatenate([rng.uniform(0.6, 0.99, 400), rng.uniform(0.01, 0.4, 400)])
labels = [True]*400 + [False]*400
print(pt.membership_inference_eval(preds.tolist(), labels, 0.5))

In [ ]:
eq = cohort.groupby(['age', 'zip_prefix', 'condition']).size().values
print(pt.reidentification_scenario(['age', 'zip_prefix', 'condition'], 50000, eq.tolist()))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.hist(eq, bins=30, color='#4f8cff', edgecolor='#161c24')
ax.axvline(1, color='#e05b5b', ls='--', label=f"{(eq == 1).mean()*100:.0f}% singletons")
ax.set_yscale('log'); ax.set_xlabel('equivalence class size'); ax.set_ylabel('classes')
ax.legend(); ax.set_title('Re-identification risk — small cohort')